# Load PPG/ABP Data and Extract Blood Pressure Labels

This notebook loads the raw `ppg` and `abp` datasets produced by `data/load_files.py` (each row is a 625-sample / 5-second window; the final window of a recording may be right-padded with NaN) and extracts systolic (SBP) and diastolic (DBP) blood pressure values from each ABP window. 

**Steps:**
1. Load `ppg` and `abp` arrays from `data/processed_dataset.npz`.
2. For each ABP window, find peaks (systolic) and valleys (diastolic) using `scipy.signal.find_peaks`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed_dataset.npz")

In [2]:
with np.load(DATA_PATH) as data:
    ppg_dataset = data["ppg"]
    abp_dataset = data["abp"]

ppg_dataset = pd.DataFrame(ppg_dataset)
abp_dataset = pd.DataFrame(abp_dataset)

print(f"ppg_dataset shape: {ppg_dataset.shape}")
print(f"abp_dataset shape: {abp_dataset.shape}")

ppg_dataset shape: (528828, 625)
abp_dataset shape: (528828, 625)


## Extract systolic and diastolic values from each ABP window

For each 625-sample ABP window: systolic pressure (SBP) is the max value in the window, and diastolic pressure (DBP) is the min value in the window, computed directly on the whole `abp_dataset` array with vectorized numpy ops (no per-row Python loop). Padded windows (containing NaN) naturally produce NaN.

In [3]:
abp_dataset["systolic pressure"] = abp_dataset.max(axis=1)
abp_dataset["diastolic pressure"] = abp_dataset.min(axis=1)

# join the 2 new columns onto the ppg dataset 
ppg_dataset["systolic pressure"] = abp_dataset["systolic pressure"]
ppg_dataset["diastolic pressure"] = abp_dataset["diastolic pressure"]
print(f"ppg_dataset shape: {ppg_dataset.shape}")

ppg_dataset shape: (528828, 627)


In [4]:
ppg_dataset

,0,1,2,3,4,5,6,7,8,9,...,617,618,619,620,621,622,623,624,systolic pressure,diastolic pressure
0,1.759531,1.718475,1.684262,1.657869,1.637341,1.615836,1.593353,1.570870,1.549365,1.526882,...,1.734115,1.698925,1.668622,1.641251,1.615836,1.591398,1.565005,1.539589,125.187439,65.597632
1,1.513196,1.485826,1.463343,1.443793,1.419355,1.393939,1.369501,1.345064,1.321603,1.300098,...,1.211144,1.188661,1.164223,1.141740,1.130010,1.141740,1.189638,1.283480,127.434268,67.307176
2,1.427175,1.584555,1.757576,1.968719,2.190616,2.399804,2.579668,2.725318,2.833822,2.910068,...,2.938416,2.957967,2.956989,2.940371,2.911046,2.868035,2.819159,2.763441,127.190048,68.381746
3,2.694037,2.611926,2.518084,2.415445,2.306940,2.196481,2.088954,1.986315,1.892473,1.807429,...,1.893451,1.815249,1.747801,1.689150,1.640274,1.605083,1.577713,1.552297,124.308245,66.037229
4,1.527859,1.504399,1.480938,1.457478,1.432063,1.405670,1.378299,1.350929,1.324536,1.304985,...,1.054741,1.038123,1.021505,1.002933,0.989247,0.975562,0.956012,0.938416,128.508838,67.551396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528823,1.165200,1.148583,1.136852,1.130010,1.127077,1.128055,1.130987,1.134897,1.137830,1.140762,...,2.327468,2.259042,2.186706,2.110459,2.040078,1.971652,1.895406,1.815249,101.986490,50.309427
528824,1.734115,1.654936,1.579668,1.511241,1.450635,1.398827,1.355816,1.323558,1.301075,1.288368,...,2.644184,2.584555,2.522972,2.448680,2.364614,2.282502,2.199413,2.114370,101.693425,50.065207
528825,2.863148,2.840665,2.810362,2.774194,2.731183,2.683284,2.631476,2.576735,2.521017,2.464321,...,2.229717,2.220919,2.217009,2.217986,2.220919,2.224829,2.226784,2.225806,112.732193,50.358272
528826,2.221896,2.217986,2.214076,2.211144,2.211144,2.214076,2.219941,2.229717,2.240469,2.253177,...,2.153470,2.071359,1.993157,1.919844,1.852395,1.790811,1.736070,1.687195,114.344048,50.993245


In [5]:
print(ppg_dataset.isnull().values.any())
print(abp_dataset.isnull().values.any())

ppg_neg_rows = {}
abp_neg_rows = {}

# ppg_neg_rows = {i:len([val for val in arr if val <= 0]) for i, arr in ppg_dataset.values if len([val for val in arr if val <= 0]) > 0} # dict is row : count of negative values
# abp_neg_rows = {i:len([val for val in arr if val <= 0]) for i, arr in abp_dataset.values if len([val for val in arr if val <= 0]) > 0}
# ppg_neg_rows = {}
for i, arr in enumerate(ppg_dataset.values):
    negs = {i:val for i, val in enumerate(arr) if val <= 0}
    if len(negs) > 0:
        ppg_neg_rows[i] = negs

for i, arr in enumerate(abp_dataset.values):
    negs = {i:val for i, val in enumerate(arr) if val <= 0}
    if len(negs) > 0:
        abp_neg_rows[i] = negs

# # ppg_neg_values = [{len([val for val in arr if val <= 0])} for arr in ppg_dataset.values if len([val for val in arr if val <= 0]) > 0]
# ppg_neg_values = [len(negs) for negs in ppg_neg_rows.values()]

False
False


In [6]:
print("PPG number of neg rows: ", len(ppg_neg_rows))
print("ABP number of neg rows: ", len(abp_neg_rows))

leniency_percent = 0.05 # 627 entries per row, if 5% entries are inadmissible (<=0, null), we will remove row from dataset
num_entries = 627
leniency = int(num_entries * leniency_percent)

PPG number of neg rows:  5407
ABP number of neg rows:  0


In [ ]:
dropped_num = 0

for row in ppg_neg_rows:
    if len(ppg_neg_rows[row]) > leniency:
        ppg_dataset.drop(row, inplace=True)
        abp_dataset.drop(row, inplace=True)
        dropped_num += 1
        print("dropped row: ", row)

print("Total rows dropped: ", dropped_num)

dropped row:  843
dropped row:  9666
dropped row:  19683
dropped row:  33442
dropped row:  60680
dropped row:  66702
dropped row:  89609
dropped row:  96165
dropped row:  103501
dropped row:  108575
dropped row:  120406
dropped row:  132910
dropped row:  133380
dropped row:  142945
dropped row:  149185
dropped row:  151494
dropped row:  162865
dropped row:  216678
dropped row:  218476
dropped row:  241182
dropped row:  246554
dropped row:  257243
dropped row:  274181


In [ ]:
print(ppg_dataset.shape)

(528775, 627)
